# 공연 예정일정 기반 분기·시간대 노출 구축
종로구·마포구, 2023~2025년 12분기의 기존 500m 시설–상권 연결에 KOPIS의 **공식 일정 문구와 러닝타임을 이용한 예정회차**를 결합한다. 실제 날짜별 회차·관객·실제 종료시각을 관측한 자료가 아니다. 현재 상세의 요일별 일정이 과거 전체 기간에 유지되었다고 가정한 탐색적 노출지표다.

종료 후 0~3시간을 주창, 종료 후 3~6시간과 0~6시간 및 **시작 전** 3시간을 별도 창으로 둔다. 3~6시간은 종료 기준이며, 시작 전 창은 종료 기준이 아니다. 창별·회차별 겹침분을 180분으로 나눈 예정회차 등가를 사용한다. 0~6시간만 360으로 나누지 않고 동일하게 180으로 나누므로 `post06 = post03 + post36`이다.

원자료·기존 분석·고정 메타데이터는 수정하지 않는다. 네트워크 요청 및 회귀모형 재적합 없이 실행한다. 최종 생성물만 `data/analysis/final/time_windows/`와 `results/main/time_windows/09_*`에 저장한다.

## 1. 고정 입력과 연구 격자
검증된 KOPIS 상세와 기존 공연–시설–분기 연결, 시설점–상권폴리곤 최소거리, 상권 목록을 읽는다. 세부상권 경계·시설 좌표는 현재 자료이며 과거 위치 불변을 보장하지 않는다. 매니페스트 전체나 인증 설정을 출력하지 않는다.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import hashlib, json, re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'metadata/main/pipeline.json').is_file())
DATA = ROOT/'data/analysis/final/time_windows'
REPORTS = ROOT/'results/main/time_windows'
DATA.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
START, STOP = datetime(2023,1,1), datetime(2026,1,1)
PAD_START, PAD_STOP = START-timedelta(days=1), STOP+timedelta(days=1)
QUARTERS = [f'{year}{quarter}' for year in range(2023,2026) for quarter in range(1,5)]
BIN_NAMES = ['00_06','06_11','11_14','14_17','17_21','21_24']
BIN_EDGES = [0,360,660,840,1020,1260,1440]
WINDOWS = ['post03','post36','post06','pre03']
input_records = {}

In [2]:
def fingerprint(relative, expected=None):
    path = ROOT/relative
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if expected is not None:
        assert actual == expected, f'고정 입력 해시 불일치: {relative}'
    input_records[relative] = dict(path=relative,sha256_before=actual,expected_sha256=expected)
    return path
manifest = json.loads(fingerprint('metadata/main/kopis_analysis_sources.json').read_text(encoding='utf-8'))
scope = json.loads(fingerprint('results/main/baseline/08_food_scope_metadata.json').read_text(encoding='utf-8'))
assert manifest['validated'] is True
assert manifest['run_id'] == scope['exposure_snapshot_linkage']['collection_run_id']
expected = {row['path']:row['sha256'] for row in scope['source_files']}
perf_spec = manifest['files']['performance_details']
performances = pd.read_csv(fingerprint(perf_spec['path'],perf_spec['sha256']),dtype=str,keep_default_na=False)
distances = pd.read_csv(fingerprint('data/analysis/final/venue_area_distances_800m.csv',expected['data/analysis/final/venue_area_distances_800m.csv']),dtype={'area_id':str,'mt10id':str})
links = pd.read_csv(fingerprint('data/analysis/final/performance_facility_quarter.csv',expected['data/analysis/final/performance_facility_quarter.csv']),dtype=str)
areas = pd.read_csv(fingerprint('data/analysis/final/study_areas.csv',expected['data/analysis/final/study_areas.csv']),dtype={'area_id':str})
assert performances.mt20id.is_unique and areas.area_id.is_unique and len(areas)==151

In [3]:
near = distances.loc[distances.distance_m.le(500) & distances.area_id.isin(areas.area_id),['area_id','mt10id']].drop_duplicates()
candidate_links = links.merge(near,on='mt10id',how='inner',validate='many_to_many')
assert candidate_links.link_method.eq('official_detail').all()
assert candidate_links.quarter.isin(QUARTERS).all()
candidate = candidate_links[['mt20id','mt10id','quarter']].drop_duplicates()
assert not candidate.duplicated(['mt20id','quarter']).any()
candidate_ids = set(candidate.mt20id)
assert len(candidate_ids)==12839
candidate_check = candidate.merge(performances[['mt20id','mt10id']],on='mt20id',how='left',suffixes=('_link','_detail'),validate='many_to_one')
assert candidate_check.mt10id_link.eq(candidate_check.mt10id_detail).all()
quarter_membership = candidate.groupby('mt20id').quarter.agg(set).to_dict()
near_map = near.groupby('mt10id').area_id.agg(list).to_dict()
print(f'500m 기존 공식 연결 공연 ID {len(candidate_ids):,}개 / 상권 {len(areas):,}개 / 분기 {len(QUARTERS)}개')

500m 기존 공식 연결 공연 ID 12,839개 / 상권 151개 / 분기 12개


**해석:** 기존 500m 노출에 기여한 12,839개 공연 ID가 후보 분모다. 다음 단계에서 일정이나 러닝타임이 해석 불가능한 ID를 노출 0으로 바꾸지 않고, 제외 사유와 상권·분기별 가용률로 보존한다.

## 2. 보수적인 일정 파서
요일 또는 순방향 요일범위 뒤 괄호 안에 시작 시각을 적은 형식만 허용한다. `HOL` 블록은 표시 여부만 보존하고 회차를 생성하지 않는다. 실제 공휴일 달력을 만들지 않으므로 공휴일에도 일반 요일 반복을 적용한 가정이다.

요일블록 중복, 동일 시작시각 중복, 역방향 요일범위, 잘못된 시각, 요일 없는 HOL 전용 일정은 제외한다. 러닝타임은 양의 단일 지속시간이고 6시간 이하여야 한다. 같은 공연 ID의 예상 상연시간이 다음 예정 시작과 겹치는 경우도 여러 일정 버전이 합쳐졌을 가능성 때문에 보수적으로 제외한다. 다른 공연끼리 같은 시각에 열리는 경우는 이 규칙으로 제외하지 않는다.

In [4]:
DAY_LABELS = ['월요일','화요일','수요일','목요일','금요일','토요일','일요일']
DAY_INDEX = {name:i for i,name in enumerate(DAY_LABELS)}
DAY_RX = '|'.join(DAY_LABELS)
BLOCK = re.compile(rf'({DAY_RX}|HOL)(?:\s*~\s*({DAY_RX}))?\(([^()]*)\)')
CLOCK = re.compile(r'(?:[01]?\d|2[0-3]):[0-5]\d')
def parse_schedule(text):
    text = str(text).strip()
    result = dict(schedule_status='ok',has_hol='HOL' in text,weekday_times={})
    if not text:
        result['schedule_status']='missing_schedule'; return result
    matches = list(BLOCK.finditer(text))
    if not matches or BLOCK.sub('',text).replace(',','').strip():
        result['schedule_status']='unsupported_schedule'; return result
    if (text[:matches[0].start()].strip() or text[matches[-1].end():].strip()
        or any(re.fullmatch(r'\s*,\s*', text[a.end():b.start()]) is None
               for a,b in zip(matches[:-1],matches[1:]))):
        result['schedule_status']='unsupported_schedule'; return result
    for match in matches:
        first,last,clocks = match.groups()
        if first=='HOL' and last is not None:
            result['schedule_status']='unsupported_schedule'; return result
        tokens = [token.strip() for token in clocks.split(',')]
        if not tokens or not all(CLOCK.fullmatch(token) for token in tokens):
            result['schedule_status']='invalid_clock'; return result
        times = [int(token.split(':')[0])*60+int(token.split(':')[1]) for token in tokens]
        if len(set(times)) != len(times):
            result['schedule_status']='duplicate_start_time'; return result
        if first=='HOL':
            continue
        left,right=DAY_INDEX[first],DAY_INDEX[last or first]
        if left>right:
            result['schedule_status']='reverse_weekday_range'; return result
        for day in range(left,right+1):
            if day in result['weekday_times']:
                result['schedule_status']='overlapping_weekday_blocks'; return result
            result['weekday_times'][day]=sorted(times)
    if not result['weekday_times']:
        result['schedule_status']='hol_only_no_normal_schedule'
    return result

In [5]:
def parse_runtime(text):
    text=str(text).strip()
    if not text:
        return np.nan,'missing_runtime'
    match=re.fullmatch(r'(?:(\d+)시간)?\s*(?:(\d+)분)?',text)
    if not match or not any(match.groups()):
        return np.nan,'unsupported_runtime'
    hours,minutes=match.groups()
    if hours and minutes and int(minutes)>=60:
        return np.nan,'noncanonical_runtime'
    value=60*int(hours or 0)+int(minutes or 0)
    if value<=0: return np.nan,'nonpositive_runtime'
    if value>360: return float(value),'runtime_over_6h'
    return float(value),'ok'
def overlaps_own_runtime(weekday_times, runtime):
    if not weekday_times or not np.isfinite(runtime): return False
    starts=sorted(day*1440+time for day,times in weekday_times.items() for time in times)
    gaps=np.diff(starts+[starts[0]+7*1440])
    return bool(np.any(gaps<runtime))

### 수동 명시한 경계 사례
정상 요일범위, 자정 시각, HOL 전용, 중복·역방향·잘못된 시각, 비정형 러닝타임과 주간 경계를 검사한다. 이 검사는 파서의 보수적 선택규칙을 확인하며 실제 일정의 진위를 검증하지 않는다.

In [6]:
edge_rows=[]
for text,expected_status in [
    ('월요일 ~ 금요일(19:30), 토요일(15:00,19:00), HOL(14:00)','ok'),
    ('월요일(00:30)','ok'),('HOL(14:00)','hol_only_no_normal_schedule'),
    ('월요일(19:00,19:00)','duplicate_start_time'),
    ('월요일(19:00), 월요일(20:00)','overlapping_weekday_blocks'),
    ('금요일 ~ 월요일(19:00)','reverse_weekday_range'),
    ('월요일(24:00)','invalid_clock'),('매일(19:00)','unsupported_schedule'),
    ('월요일(19:00),,화요일(20:00)','unsupported_schedule'),
    ('HOL ~ 월요일(19:00)','unsupported_schedule')]:
    actual=parse_schedule(text)['schedule_status']
    edge_rows.append(dict(test='schedule',input=text,expected=expected_status,actual=actual,passed=actual==expected_status))
for text,expected_status in [('1시간 30분','ok'),('85분','ok'),('1시간 60분','noncanonical_runtime'),('7시간','runtime_over_6h')]:
    actual=parse_runtime(text)[1]
    edge_rows.append(dict(test='runtime',input=text,expected=expected_status,actual=actual,passed=actual==expected_status))
assert overlaps_own_runtime({6:[23*60+30],0:[30]},90)
assert not overlaps_own_runtime({0:[19*60],1:[19*60]},90)
assert all(row['passed'] for row in edge_rows)
display(pd.DataFrame(edge_rows))

,test,input,expected,actual,passed
0,schedule,"월요일 ~ 금요일(19:30), 토요일(15:00,19:00), HOL(14:00)",ok,ok,True
1,schedule,월요일(00:30),ok,ok,True
2,schedule,HOL(14:00),hol_only_no_normal_schedule,hol_only_no_normal_schedule,True
3,schedule,"월요일(19:00,19:00)",duplicate_start_time,duplicate_start_time,True
4,schedule,"월요일(19:00), 월요일(20:00)",overlapping_weekday_blocks,overlapping_weekday_blocks,True
5,schedule,금요일 ~ 월요일(19:00),reverse_weekday_range,reverse_weekday_range,True
6,schedule,월요일(24:00),invalid_clock,invalid_clock,True
7,schedule,매일(19:00),unsupported_schedule,unsupported_schedule,True
8,schedule,"월요일(19:00),,화요일(20:00)",unsupported_schedule,unsupported_schedule,True
9,schedule,HOL ~ 월요일(19:00),unsupported_schedule,unsupported_schedule,True


In [7]:
audit_rows=[]; schedules={}
for row in performances.itertuples(index=False):
    parsed=parse_schedule(row.dtguidance)
    runtime,runtime_status=parse_runtime(row.prfruntime)
    start=pd.to_datetime(row.prfpdfrom,format='%Y.%m.%d',errors='coerce')
    stop=pd.to_datetime(row.prfpdto,format='%Y.%m.%d',errors='coerce')
    period_ok=bool(pd.notna(start) and pd.notna(stop) and start<=stop)
    overlap=overlaps_own_runtime(parsed['weekday_times'],runtime)
    ok=parsed['schedule_status']=='ok' and runtime_status=='ok' and period_ok and not overlap
    schedules[row.mt20id]=parsed['weekday_times']
    audit_rows.append(dict(mt20id=row.mt20id,mt10id=row.mt10id,dtguidance=row.dtguidance,
        prfruntime=row.prfruntime,prfpdfrom=row.prfpdfrom,prfpdto=row.prfpdto,
        schedule_status=parsed['schedule_status'],runtime_status=runtime_status,runtime_minutes=runtime,
        has_hol=parsed['has_hol'],own_runtime_overlap_ambiguous=overlap,valid_period=period_ok,
        period_extends_beyond_2025=bool(period_ok and stop>=STOP),
        schedule_runtime_eligible=ok,in_500m_candidate=row.mt20id in candidate_ids))
audit=pd.DataFrame(audit_rows)
display(audit.loc[audit.in_500m_candidate].groupby(['schedule_status','runtime_status','own_runtime_overlap_ambiguous']).size().rename('공연ID수').reset_index())

,schedule_status,runtime_status,own_runtime_overlap_ambiguous,공연ID수
0,hol_only_no_normal_schedule,missing_runtime,False,4
1,hol_only_no_normal_schedule,ok,False,79
2,missing_schedule,ok,False,7
3,ok,missing_runtime,False,445
4,ok,noncanonical_runtime,False,1
5,ok,ok,False,12058
6,ok,ok,True,102
7,ok,runtime_over_6h,False,49
8,ok,runtime_over_6h,True,1
9,overlapping_weekday_blocks,missing_runtime,False,2


**해석:** 많은 시작시각이 적힌 장기공연이 반드시 그 시각마다 실제 공연되었다고 볼 수 없다. 러닝타임과 겹치는 시간표를 제외한 규칙은 과대 회차 생성을 줄이는 대신 특정 공연을 선택적으로 제외할 수 있다. 따라서 전체 가용률과 90%·100% 표본을 뒤에서 함께 제공한다.

## 3. 날짜별 예정 시작·예상 종료 생성
연구기간에 하루를 앞뒤로 더해 2022-12-31~2026-01-01의 예정 시작을 순회한다. 연구기간 안에서는 기존 공연–분기 membership도 만족해야 한다. 범위 밖 하루는 같은 ID가 첫·마지막 연구분기의 후보일 때만 허용한다. 연구기간 목록에 전혀 없는 2022년 말·2026년 초 공연은 여전히 누락될 수 있다.

기간 안에 일반 요일 반복과 일치하는 예정 시작이 하나도 없는 후보도 0회차로 확정하지 않고 가용하지 않은 ID로 남긴다. 이 달력 반복은 HOL 특별일정, 휴연, 특별회차, 과거 변경 내역을 복원하지 않는다.

In [8]:
def quarter_of(value):
    return f'{value.year}{(value.month-1)//3+1}'
def quarter_bounds(quarter):
    year,q=int(quarter[:4]),int(quarter[-1]); month=3*q-2
    start=datetime(year,month,1)
    end=datetime(year+1,1,1) if q==4 else datetime(year,month+3,1)
    return start,end
session_rows=[]; generated_counts=Counter()
for row in audit.loc[audit.in_500m_candidate & audit.schedule_runtime_eligible].itertuples(index=False):
    first=max(datetime.strptime(row.prfpdfrom,'%Y.%m.%d'),PAD_START)
    last=min(datetime.strptime(row.prfpdto,'%Y.%m.%d'),PAD_STOP-timedelta(days=1))
    days=pd.date_range(first,last,freq='D') if first<=last else []
    allowed=quarter_membership[row.mt20id]
    for day in days:
        q=quarter_of(day)
        allowed_day=(q in allowed) or (day<START and '20231' in allowed) or (day>=STOP and '20254' in allowed)
        if not allowed_day: continue
        for minutes in schedules[row.mt20id].get(day.weekday(),[]):
            nominal_start=day.to_pydatetime()+timedelta(minutes=minutes)
            estimated_end=nominal_start+timedelta(minutes=row.runtime_minutes)
            session_rows.append((row.mt20id,row.mt10id,nominal_start,estimated_end,q,day<START or day>=STOP))
            generated_counts[row.mt20id]+=1
sessions=pd.DataFrame(session_rows,columns=['mt20id','mt10id','nominal_start','estimated_end','start_quarter','padding_start'])
audit['nominal_sessions_generated']=audit.mt20id.map(generated_counts).fillna(0).astype(int)
audit['exposure_eligible']=audit.in_500m_candidate & audit.schedule_runtime_eligible & audit.nominal_sessions_generated.gt(0)
audit['no_matching_normal_occurrence']=audit.in_500m_candidate & audit.schedule_runtime_eligible & audit.nominal_sessions_generated.eq(0)
assert not sessions.duplicated(['mt20id','nominal_start']).any()
print(f'생성한 예정회차 {len(sessions):,}개 / 기간 밖 하루 시작 {sessions.padding_start.sum():,}개 / 유효 공연 ID {sessions.mt20id.nunique():,}개')

생성한 예정회차 135,362개 / 기간 밖 하루 시작 289개 / 유효 공연 ID 12,047개


## 4. 창의 자정·분기·판매구간 겹침
창은 왼쪽 포함·오른쪽 제외 구간이다. 매출의 고정 구간은 00~06, 06~11, 11~14, 14~17, 17~21, 21~24시다. 날짜가 바뀌면 새로운 날짜·분기에 귀속하고, 연구범위 밖의 겹침도 보존해 보존식을 검증한다.

같은 시간에 여러 예정회차 창이 겹치면 **회차별로 합산**한다. 합집합으로 잘라 중복을 지우지 않으므로 지표가 구간 길이나 1을 넘을 수 있다. 이 값은 고유한 관객 체류시간·고유 관객 수가 아니다.

In [9]:
def split_window(begin,end):
    pieces=[]; day=begin.replace(hour=0,minute=0,second=0,microsecond=0)
    while day<end:
        for index,(left,right) in enumerate(zip(BIN_EDGES[:-1],BIN_EDGES[1:])):
            a=max(begin,day+timedelta(minutes=left)); b=min(end,day+timedelta(minutes=right))
            if a<b: pieces.append((quarter_of(day),index,int((b-a).total_seconds()//60)))
        day+=timedelta(days=1)
    assert sum(piece[2] for piece in pieces)==int((end-begin).total_seconds()//60)
    return pieces
cross=split_window(datetime(2023,3,31,23,30),datetime(2023,4,1,2,30))
assert cross==[('20231',5,30),('20232',0,150)]
midnight=split_window(datetime(2023,12,31,23),datetime(2024,1,1,5))
assert midnight==[('20234',5,60),('20241',0,300)]
pre=split_window(datetime(2022,12,31,23),datetime(2023,1,1,2))
assert pre==[('20224',5,60),('20231',0,120)]
edge_rows += [dict(test='window',input='quarter_midnight_split',expected='30+150',actual='30+150',passed=True),
              dict(test='window',input='year_boundary_split',expected='60+300',actual='60+300',passed=True),
              dict(test='window',input='pre_start_boundary',expected='60+120',actual='60+120',passed=True)]

In [10]:
minute_totals=defaultdict(lambda:np.zeros(4,dtype=np.int64))
for row in sessions.itertuples(index=False):
    intervals=[(row.estimated_end,row.estimated_end+timedelta(hours=3)),
               (row.estimated_end+timedelta(hours=3),row.estimated_end+timedelta(hours=6)),
               (row.estimated_end,row.estimated_end+timedelta(hours=6)),
               (row.nominal_start-timedelta(hours=3),row.nominal_start)]
    for window_index,(begin,end) in enumerate(intervals):
        for q,bin_index,minutes in split_window(begin,end):
            minute_totals[(row.mt20id,row.mt10id,q,bin_index)][window_index]+=minutes
minute_rows=[(*key,*values) for key,values in minute_totals.items()]
perf_minutes=pd.DataFrame(minute_rows,columns=['mt20id','mt10id','quarter','bin_index',*[f'minutes_{w}' for w in WINDOWS]])
assert np.array_equal(perf_minutes.minutes_post06,perf_minutes.minutes_post03+perf_minutes.minutes_post36)

In [11]:
conservation=[]
for w,multiplier in [('post03',1),('post36',1),('post06',2),('pre03',1)]:
    actual=int(perf_minutes[f'minutes_{w}'].sum())
    expected_minutes=len(sessions)*180*multiplier
    assert actual==expected_minutes
    inside=int(perf_minutes.loc[perf_minutes.quarter.isin(QUARTERS),f'minutes_{w}'].sum())
    conservation.append(dict(level='performance_session',window=w,expected_minutes=expected_minutes,
        study_minutes=inside,outside_minutes=actual-inside,actual_minutes=actual,passed=actual==expected_minutes))
print('공연 예정회차별 네 창의 분 합계와 post06=post03+post36 보존식 통과')

공연 예정회차별 네 창의 분 합계와 post06=post03+post36 보존식 통과


## 5. 창 경계를 반영한 가용률
원분기 후보수와 창의 영향을 받을 수 있는 후보수는 구분한다. 원분기 첫날에 등록기간이 걸치면 시작 전 창이 이전분기에, 마지막 날에 걸치면 종료 후 창이 다음분기에 영향을 줄 수 있으므로 후보 ID를 인접분기에도 보수적으로 포함한다. 중복 ID는 합집합으로 센다.

`availability_pct`는 확장된 창 의존 후보 중 일정·러닝타임·기간·실제 생성 조건을 충족한 ID 비율이다. 회차 수나 관객 수에 가중한 가용률이 아니다. 후보가 0개인 경우 관측 목록 내 등록 공연 없음으로 100% 가용·노출 0을 허용한다. 알려진 부분합은 `known_exposure_*`, 후보가 모두 가용한 경우에만 값이 있는 완전표본 필드는 `exposure_*`다. 파싱 실패가 있는 분기의 완전표본 필드는 NaN으로 둔다.

In [12]:
audit_index=audit.set_index('mt20id')
source_sets=defaultdict(set); dependency_sets=defaultdict(set)
for row in candidate_links.itertuples(index=False):
    source_sets[(row.area_id,row.quarter)].add(row.mt20id)
for row in candidate.itertuples(index=False):
    q_start,q_stop=quarter_bounds(row.quarter)
    record=audit_index.loc[row.mt20id]
    period_start=datetime.strptime(record.prfpdfrom,'%Y.%m.%d')
    period_stop=datetime.strptime(record.prfpdto,'%Y.%m.%d')
    dependent={row.quarter}
    if period_start<=q_start<=period_stop: dependent.add(quarter_of(q_start-timedelta(days=1)))
    if period_start<=q_stop-timedelta(days=1)<=period_stop: dependent.add(quarter_of(q_stop))
    for area_id in near_map[row.mt10id]:
        for q in dependent.intersection(QUARTERS): dependency_sets[(area_id,q)].add(row.mt20id)
eligible_ids=set(audit.loc[audit.exposure_eligible,'mt20id'])

In [13]:
availability_rows=[]
for area_id in sorted(areas.area_id):
    for q in QUARTERS:
        ids=dependency_sets[(area_id,q)]; selected=audit_index.loc[sorted(ids)]
        n=len(ids); valid=len(ids.intersection(eligible_ids))
        availability_rows.append(dict(area_id=area_id,quarter=q,
            candidate_performance_ids_source_quarter=len(source_sets[(area_id,q)]),
            candidate_performance_ids_window_dependency=n,
            eligible_performance_ids_window_dependency=valid,
            availability_pct=100*valid/n if n else 100.0,
            complete_schedule_coverage=valid==n,
            excluded_schedule_ids=int(selected.schedule_status.ne('ok').sum()),
            excluded_runtime_ids=int(selected.runtime_status.ne('ok').sum()),
            overlap_ambiguous_ids=int(selected.own_runtime_overlap_ambiguous.sum()),
            no_matching_normal_occurrence_ids=int(selected.no_matching_normal_occurrence.sum()),
            has_hol_candidate_ids=int(selected.has_hol.sum()),
            period_extends_beyond_2025_ids=int(selected.period_extends_beyond_2025.sum()),
            study_endpoint_quarter=q in ['20231','20254']))
availability=pd.DataFrame(availability_rows)
assert len(availability)==151*12
assert (availability.eligible_performance_ids_window_dependency<=availability.candidate_performance_ids_window_dependency).all()
display(availability[['availability_pct','candidate_performance_ids_window_dependency']].describe())

,availability_pct,candidate_performance_ids_window_dependency
count,1812.000000,1812.000000
mean,92.306643,58.257174
std,15.598567,96.431847
min,0.000000,0.000000
25%,90.393640,1.000000
50%,97.972880,15.500000
75%,100.000000,60.000000
max,100.000000,627.000000


**해석:** 모형 주표본은 결과를 보기 전 합의한 가용률 90% 이상이며 알려진 부분 노출을 사용한다. 100% 가용표본은 별도 민감도다. 부분표본의 알려진 노출이 0이라고 해서 미확인 공연까지 없었다고 해석할 수 없다. HOL과 장기공연 포함도 가용률과 별개의 한계다.

## 6. 상권·분기·판매시간대 결합용 표
기존 시설–상권 500m 관계를 적용한다. 하나의 공연장이 여러 상권에 연결될 수 있으므로 같은 예정회차 노출도 여러 상권에 공유된다. 업종별 노출은 동일 상권·분기·시간대 내에서 같으며, 이후 10종 매출표와 키로 결합한다.

In [14]:
area_minutes=perf_minutes.merge(near,on='mt10id',how='inner',validate='many_to_many')
weighted_sessions=int(sessions.mt10id.map(lambda x:len(near_map[x])).sum())
for w,multiplier in [('post03',1),('post36',1),('post06',2),('pre03',1)]:
    actual=int(area_minutes[f'minutes_{w}'].sum()); expected_minutes=weighted_sessions*180*multiplier
    assert actual==expected_minutes
    inside=int(area_minutes.loc[area_minutes.quarter.isin(QUARTERS),f'minutes_{w}'].sum())
    conservation.append(dict(level='shared_area_session',window=w,expected_minutes=expected_minutes,
        study_minutes=inside,outside_minutes=actual-inside,actual_minutes=actual,passed=True))
keep=area_minutes.loc[area_minutes.quarter.isin(QUARTERS)]
# 실제 계산 창이 가용률의 후보 합집합에 포함되었는지 확인한다.
for (area_id,q),part in keep.groupby(['area_id','quarter']):
    assert set(part.mt20id).issubset(dependency_sets[(area_id,q)])
grouped=keep.groupby(['area_id','quarter','bin_index'])[[f'minutes_{w}' for w in WINDOWS]].sum()
grid=pd.MultiIndex.from_product([sorted(areas.area_id),QUARTERS,range(6)],names=['area_id','quarter','bin_index'])
exposure=grouped.reindex(grid,fill_value=0).reset_index()
exposure['time_bin']=exposure.bin_index.map(dict(enumerate(BIN_NAMES)))
exposure=exposure.merge(availability,on=['area_id','quarter'],validate='many_to_one')

In [15]:
for w in WINDOWS:
    exposure[f'known_exposure_{w}']=exposure[f'minutes_{w}']/180
    exposure[f'exposure_{w}']=exposure[f'known_exposure_{w}'].where(exposure.complete_schedule_coverage)
assert np.array_equal(exposure.minutes_post06,exposure.minutes_post03+exposure.minutes_post36)
assert np.allclose(exposure.known_exposure_post06,exposure.known_exposure_post03+exposure.known_exposure_post36)
assert exposure.loc[~exposure.complete_schedule_coverage,[f'exposure_{w}' for w in WINDOWS]].isna().all().all()
assert not exposure.duplicated(['area_id','quarter','time_bin']).any()
assert len(exposure)==151*12*6
exposure=exposure[['area_id','quarter','time_bin','bin_index',*[f'known_exposure_{w}' for w in WINDOWS],
    *[f'exposure_{w}' for w in WINDOWS],*[f'minutes_{w}' for w in WINDOWS],
    'availability_pct','candidate_performance_ids_source_quarter','candidate_performance_ids_window_dependency',
    'eligible_performance_ids_window_dependency','complete_schedule_coverage','study_endpoint_quarter']]
display(exposure.head(6))

,area_id,quarter,time_bin,bin_index,known_exposure_post03,known_exposure_post36,known_exposure_post06,known_exposure_pre03,exposure_post03,exposure_post36,...,minutes_post03,minutes_post36,minutes_post06,minutes_pre03,availability_pct,candidate_performance_ids_source_quarter,candidate_performance_ids_window_dependency,eligible_performance_ids_window_dependency,complete_schedule_coverage,study_endpoint_quarter
0,3001494,20231,00_06,0,59.694444,376.250000,435.944444,0.000000,NaN,NaN,...,10745,67725,78470,0,90.977444,130,133,121,False,True
1,3001494,20231,06_11,1,0.000000,0.000000,0.000000,157.000000,NaN,NaN,...,0,0,0,28260,90.977444,130,133,121,False,True
2,3001494,20231,11_14,2,107.833333,0.000000,107.833333,284.500000,NaN,NaN,...,19410,0,19410,51210,90.977444,130,133,121,False,True
3,3001494,20231,14_17,3,181.611111,107.833333,289.444444,158.833333,NaN,NaN,...,32690,19410,52100,28590,90.977444,130,133,121,False,True
4,3001494,20231,17_21,4,240.222222,287.666667,527.888889,309.666667,NaN,NaN,...,43240,51780,95020,55740,90.977444,130,133,121,False,True
5,3001494,20231,21_24,5,319.388889,134.166667,453.555556,0.000000,NaN,NaN,...,57490,24150,81640,0,90.977444,130,133,121,False,True


## 7. 결과 저장과 최종 검증
예정회차 원장, 키 결합용 노출, 가용률, 공연 ID별 제외 사유, 수동 경계검사, 창 보존검사, 선택규칙·한계를 저장한다. 노출 필드의 단위와 결측 의미를 메타데이터에 포함한다. 입력파일은 실행 후에도 같은 해시여야 한다.

In [16]:
outputs={
    DATA/'area_quarter_timebin_exposure.csv':exposure,
    DATA/'availability.csv':availability,
    DATA/'nominal_sessions.csv':sessions,
    REPORTS/'09_performance_schedule_audit.csv':audit,
    REPORTS/'09_parser_edge_checks.csv':pd.DataFrame(edge_rows),
    REPORTS/'09_window_conservation.csv':pd.DataFrame(conservation)}
for path,frame in outputs.items(): frame.to_csv(path,index=False,encoding='utf-8-sig')
for relative,record in input_records.items():
    record['sha256_after']=hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()
    record['unchanged']=record['sha256_before']==record['sha256_after']
assert all(row['unchanged'] for row in input_records.values())

In [17]:
summary={
    'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'clock_timezone':'Asia/Seoul','kopis_collection_run_id':manifest['run_id'],
    'study_period':['2023-01-01','2025-12-31'],'nominal_start_padding_days':1,'radius_m':500,
    'area_count':151,'quarters':QUARTERS,'time_bins':BIN_NAMES,'candidate_performance_ids':len(candidate_ids),
    'eligible_performance_ids':len(eligible_ids),'nominal_sessions_including_padding':len(sessions),
    'nominal_sessions_study_start':int((~sessions.padding_start).sum()),
    'nominal_sessions_padding_start':int(sessions.padding_start.sum()),
    'area_quarters_total':len(availability),'area_quarters_availability_ge90':int(availability.availability_pct.ge(90).sum()),
    'area_quarters_availability_100':int(availability.complete_schedule_coverage.sum()),
    'area_quarters_zero_candidates':int(availability.candidate_performance_ids_window_dependency.eq(0).sum()),
    'availability_denominator':'distinct candidate performance IDs that may affect current/adjacent quarter windows, unioned per area-quarter',
    'source_quarter_denominator':'original list-derived performance-facility-quarter membership',
    'primary_analysis_threshold_pct':90,'sensitivity_complete_threshold_pct':100,
    'exposure_unit':'nominal-session-equivalent = summed per-session window overlap minutes / 180',
    'window_definition':{'post03':'[estimated_end,estimated_end+3h)','post36':'[estimated_end+3h,estimated_end+6h)',
        'post06':'[estimated_end,estimated_end+6h)','pre03':'[nominal_start-3h,nominal_start)'},
    'post06_normalization_minutes':180,'post06_equals_post03_plus_post36':True,
    'overlap_rule':'sum each nominal session separately; no union/deduplication across different sessions or performance IDs',
    'known_exposure_fields':'partial sums of eligible schedules, not total actual exposure',
    'exposure_fields':'known sums only where all window-dependent candidate IDs are eligible, otherwise missing',
    'zero_candidate_rule':'availability=100 and known/complete exposure=0 for no registered candidate in fixed observed list',
    'normal_weekday_recurrence_assumption':True,'holiday_calendar_used':False,'HOL_blocks_generated':False,
    'actual_session_data':False,'actual_end_time_data':False,'historical_schedule_verified':False,
    'runtime_max_minutes':360,'exclude_within_ID_runtime_overlap':True,
    'study_boundary_limitation':'IDs entirely outside 2023–2025 list may contribute boundary windows but cannot be recovered; study endpoint quarters flagged',
    'limitations':['current schedule is repeated over historical registered period; actual date-specific changes unknown',
        'normal weekday pattern used even on public holidays; HOL special schedules omitted',
        'runtime predicts ending; delays/intermissions/cancellations not observed',
        'strict exclusions cause selective schedule coverage; ID coverage is not audience or session-weighted coverage',
        'current facility coordinates and area boundaries applied to historical quarters',
        'multiple areas share performance exposure; distinct-area spatial dependence remains',
        'quarterly fixed-time-bin sales cannot identify actual event-date post-performance spending'],
    'network_requests':0,'new_model_fits':0,'all_inputs_unchanged':True,'input_records':list(input_records.values()),
    'outputs':{path.relative_to(ROOT).as_posix():{'rows':len(frame),'columns':frame.columns.tolist(),
        'sha256':hashlib.sha256(path.read_bytes()).hexdigest()} for path,frame in outputs.items()}}
(REPORTS/'09_exposure_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'완료: 노출 {len(exposure):,}행, 가용률 {len(availability):,}행, 예정회차 {len(sessions):,}행')
print(f'상권·분기 가용률 90% 이상 {summary["area_quarters_availability_ge90"]:,}개 / 100% {summary["area_quarters_availability_100"]:,}개')

완료: 노출 10,872행, 가용률 1,812행, 예정회차 135,362행
상권·분기 가용률 90% 이상 1,381개 / 100% 861개


**최종 해석:** 이 자료는 일반 요일의 예정 일정에 기초한 분기별 시간대 노출이며 실제 공연 종료 후 매출을 직접 측정한 결과가 아니다. 파싱 실패나 일정 모호성을 공연 부재로 바꾸지 않았다. 모형은 알려진 부분 노출과 사전 설정한 가용률 기준을 사용하고, 완전표본·자료 한계와 함께 해석해야 한다.